📝
# FieldCare — Sprint 1 Campus notebook

HelioDesk has an AI application that combines equipment evidence and a model. Dispatch wants to call that work as a service; a supervisor later needs a different briefing. Build that progression in **this one notebook**.

By the end, you will have called and traced a FastAPI service, added a route with its own prompt, applied input/output contracts, kept two route versions independent, and built a new supervisor operation. Your final personal checkpoint ZIP becomes the starting point for the next sprint.

## How to use this notebook

Read each short explanation, predict the result, then run its code cell. In Colab, use the left-hand table of contents to resume at a section. **Worked example** cells show a complete pattern. **Your turn** cells leave the design to you. **Checkpoint** cells test observations; they do not award a grade.

| Section | What you do | What you keep |
|---|---|---|
| 0. Setup | Prepare the runtime once | One personal project and one worked-example lab |
| 1. Call and trace | Inspect the HTTP exchange and adapter boundary | Request/response and trace observations |
| 2. Configure a route | Follow a supplied analogous route | A pattern you can adapt |
| 3. Dispatch handover | Make an independent prompt/endpoint choice | Your first added route |
| 4. Contracts | Observe validation before/after work | A concrete input/output agreement |
| 5. Versions | Observe independent bindings, then apply a pilot contract | Preserved `/v1` and a new `/v2` |
| 6. Supervisor checkpoint | Combine the taught patterns independently | A new versioned operation and evidence |
| 7. Save | Export once and download | One source/evidence ZIP plus your saved notebook |

**Run-all behavior:** worked examples execute in demo mode. Unfilled exercises report `pending`; running all cells does not complete them. Complete a task, then rerun its check cells. Provider calls are off unless you explicitly enable the live comparison and enter your own key privately.

**Before starting:** open this notebook from the course GitHub repository, save a copy to Drive, select a Python CPU runtime, and run setup. Setup downloads the shared code at the configured release revision and installs it. Locally, open this notebook from your cloned course repository. Never paste credentials into source, notebook cells, outputs or chat.

Use this Campus notebook for every Campus lesson in this sprint. The live sessions use a separate Live notebook. Save one personal copy of each; keep your cumulative project here.


## Campus lessons in Notion

- [Application vs service](https://app.notion.com/p/3ea5a55972ad81a09e8feba7b4bb576a?pvs=204)
- [REST in plain language](https://app.notion.com/p/3ea5a55972ad813b906bd2f6edb69740?pvs=204)
- [Touring the provided FastAPI service](https://app.notion.com/p/3ea5a55972ad81cd844dccbe5d8b0b89?pvs=204)
- [Extending the scaffold for a new use case](https://app.notion.com/p/3ea5a55972ad811e9461cc2e184edeb6?pvs=204)
- [Why contracts matter for AI services](https://app.notion.com/p/3ea5a55972ad812c8fcaf9462d1a2b9e?pvs=204)
- [Why and how to version AI endpoints](https://app.notion.com/p/3ea5a55972ad815795d0e07af30135e7?pvs=204)
- [Service scaffold checkpoint](https://app.notion.com/p/3ea5a55972ad8180aabdcc14a959c3d2?pvs=204)


## 0. Prepare the runtime and your workspaces

Run setup once after a runtime restart. If switching source versions in an already-used kernel, restart the kernel first so imported helpers match the selected source revision. The shared repository provides the libraries and synthetic fixtures; no deployed service or provider account is needed for the local examples.

The course release will include its GitHub URL and fixed revision in setup. Those publication values are not configured in this local authoring copy yet. Existing local checkouts work without them; a fresh Colab runtime needs both values.

In [ ]:
#@title Load the shared course code from GitHub
from pathlib import Path
import os, re, sys, subprocess, tempfile

# Reviewed shared helpers: this immutable revision keeps every lesson on the same source.
GITHUB_REPO = 'https://github.com/richhiey/ai-app-dev_Mod-B.git'
GITHUB_REF = 'fb9d95b1cadcc29972cb9dd7388201aaa5868e1d'
repo_url = os.environ.get('MODULE_B_REPO_URL', GITHUB_REPO).strip()
repo_ref = os.environ.get('MODULE_B_REPO_REF', GITHUB_REF).strip()
destination = Path(os.environ.get('MODULE_B_REPO', '/content/ai-app-dev_Mod-B')).expanduser()
candidates = [destination, Path.cwd(), *Path.cwd().resolve().parents]
REPO = next((p.resolve() for p in candidates if (p/'src/module_b/notebook.py').is_file()), None)

if REPO is None:
    if not re.fullmatch(r'https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?', repo_url):
        raise RuntimeError('The course author must set GITHUB_REPO to the published HTTPS GitHub repository URL.')
    if not re.fullmatch(r'[0-9a-fA-F]{40}|refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*', repo_ref):
        raise RuntimeError('The course author must set GITHUB_REF to a release tag (refs/tags/...) or full commit SHA.')
    if destination.exists():
        raise FileExistsError('The destination already exists without the expected helpers. Preserve it and use a fresh runtime or a new MODULE_B_REPO path.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Prepare separately, so a failed download cannot replace existing student work.
    with tempfile.TemporaryDirectory(prefix='module-b-download-', dir=destination.parent) as temporary:
        checkout = Path(temporary)/'checkout'
        subprocess.run(['git','clone','--no-checkout','--depth','1',repo_url,str(checkout)],check=True)
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin',repo_ref],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
        if not (checkout/'src/module_b/notebook.py').is_file() or not (checkout/'requirements.lock').is_file():
            raise RuntimeError('The selected repository revision does not contain the expected Module B source layout.')
        checkout.rename(destination)
    REPO = destination.resolve()

# Existing local checkouts are reused; setup never pulls, resets or overwrites them.
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'-c',str(REPO/'requirements.lock')],check=True)
if str(REPO/'src') not in sys.path:
    sys.path.insert(0,str(REPO/'src'))
from module_b import __version__
print('Shared package ready:', __version__)
print('Source checkout:', REPO)

In [ ]:
import json
import httpx
from module_b.workspace import prepare_example, restore_workspace
from module_b.data import load_fixture, load_json
from module_b.source import source_excerpt
from module_b.edits import write_source
from module_b.fieldcare import demo_service, trace_paths, verify_prompt_binding, inspect_route_bindings
from module_b.contracts import inspect_contract
from module_b.versioning import inspect_version_bindings
from module_b.notebook import (table, apply_student_edits, register_source, baseline,
    observe_change, revision, fresh, provider_service, export_progress, preserved)

### Keep demonstration files separate from your project

`lab` holds complete worked examples. `project` holds your own growing service. They share the same source repository, but changing a lab does not solve your project task. There is no need to manage a notebook for each activity.

Leave `RESUME_ZIP` empty on your first visit. To resume in a new Colab runtime, upload your previously exported checkpoint using the Files panel and set its path here. Rerunning this cell retains existing work; it never resets your files. Change `RUN_NAME` only to deliberately start a separate attempt.

In [ ]:
RUN_NAME = 'sprint-1'
RESUME_ZIP = ''  # Example: /content/sprint-1-checkpoint-....zip
work = REPO/'work'/RUN_NAME
project = work/'project'
if RESUME_ZIP and not project.exists():
    project = restore_workspace(RESUME_ZIP, project)
project = prepare_example(destination=project, repo_root=REPO)
lab = prepare_example(destination=work/'lab', repo_root=REPO)
accepted = load_fixture(project, 'valid-request')
incomplete = load_fixture(project, 'incomplete-request')
initial = baseline(project, 'sprint1-entry', ['/v1/diagnose'])
evidence = {}
print('Your project:', project)
print('Worked-example lab:', lab)

## 1. Call the service and follow the request

An **application** is used directly to accomplish a task; a **service** exposes an operation another system can call. Here the notebook is the caller. FastAPI is the service wrapper; the evidence and model logic remain inside FieldCare.

An **endpoint** is an operation at a path. An HTTP **method** says what kind of request is being made; a JSON **body** carries inputs. A **status code** summarizes the outcome. Predict the difference between checking readiness with `GET /health` and asking a question with `POST /v1/diagnose`.

The `with` block starts an owned local server and stops it afterward, including if a check fails. `200` on this complete buffered diagnostic response means the HTTP operation succeeded; inspect the body to distinguish an answer from a clarification. It does not establish answer quality. `mode='demo'` identifies fixed teaching output, not a generated model answer.

In [ ]:
with demo_service(project) as service:
    health = httpx.get(service.base_url+'/health')
    response = httpx.post(service.base_url+'/v1/diagnose', json=accepted)
    evidence['first_http'] = {'health_status':health.status_code, 'status':response.status_code, 'body':response.json()}
    print('Health:', health.status_code, health.json())
    print('Request:', accepted)
    print('Response:', response.status_code, json.dumps(response.json(), indent=2))
    assert health.status_code == response.status_code == 200
    assert response.json()['mode'] == 'demo'

### Worked example — locate each boundary in actual source

A **schema** declares the shape and constraints of data. The route accepts `DiagnosticRequest` and returns `DiagnosticResponse`. An **adapter** connects the application to its model provider or the demo response.

Read the route, then the request/response declarations. In the service function, find where evidence is assembled and `generate_answer` is called. In the adapter, distinguish the fixed demo branch from the provider HTTP call. Source inspection does not run a model.

**Check:** identify the registered path, required input, optional context, four response fields and adapter call. The provider credential belongs to the service runtime, not the caller's request body.

In [ ]:
print(source_excerpt(project, 'app/routes.py', 'diagnose'))
print(source_excerpt(project, 'app/schemas.py'))
print(source_excerpt(project, 'app/service.py', 'run_diagnosis'))
print(source_excerpt(project, 'app/model.py', 'generate_answer'))

### Predict where an incomplete or invalid request stops

An absent required `question` is malformed input. A valid question without equipment context is different: the service can ask for clarification. Send both, then inspect a **separate in-process probe**, which counts function entries on new requests. It is not a counter attached to your previous HTTP calls.

Expect invalid input `422`, valid missing context `200` with `needs_clarification`, and no adapter work for either. A supported complete request enters the demo adapter. A successful status does not establish that equipment is safe or a repair is authorized.

In [ ]:
with demo_service(project) as service:
    for label, payload in [('invalid', load_fixture(project,'invalid-request')), ('missing_context', incomplete)]:
        response = httpx.post(service.base_url+'/v1/diagnose', json=payload)
        print(label, response.status_code, response.json()['detail'] if response.status_code==422 else response.json()['status'])
evidence['trace'] = trace_paths(project)
table(evidence['trace'])

## 2. Worked example — configure and register another route

Dispatch will eventually need its own prompt. First rehearse the mechanism in `lab`: a supplied route requests a three-bullet technician response at `/practice/diagnose-brief`. It keeps the existing schemas, evidence and adapter.

Read the supplied source, write it into the lab, and register its router. **Registration** connects the route definitions to the application. The explicit `include_router` line is what makes the operation callable; a file alone is not a route.

**Expected:** both the original and practice paths return `200`. Demo wording stays fixed. A separate mocked-provider check proves that the two routes supplied different prompts; it cannot judge real wording.

In [ ]:
pattern = (REPO/'examples/patterns/practice_routes.py').read_text()
print(pattern)
write_source(lab, 'app/practice_routes.py', pattern)
registration = '\nfrom app.practice_routes import router as practice_router\napp.include_router(practice_router)\n'
register_source(lab, registration)
print(registration)

In [ ]:
with demo_service(lab) as service:
    for path in ['/v1/diagnose', '/practice/diagnose-brief']:
        response = httpx.post(service.base_url+path, json=accepted)
        print(path, response.status_code)
        assert response.status_code == 200
print(verify_prompt_binding(lab))  # This helper raises if a binding check fails.

## 3. Your turn — a useful dispatch handover

✂️ **Independent practice**

Dispatch needs equipment context, the reported question, the relevant documented checks and what the next colleague must verify. Create `POST /v1/dispatch-handover` in `app/handover_routes.py`, then register it in `app/main.py`.

Adapt the provided pattern; choose the prompt wording, order and format yourself. Distinguish reported symptoms from confirmed findings. Keep the original route, schemas, shared service, adapter and evidence unchanged. Do not hardcode an answer or add output fields.

Use the file editor at the printed `project` path, or put complete file contents into `handover_edits` below. An empty dictionary preserves files and leaves the task unfinished. To revise with the dictionary, replace its text and rerun the cell. Use the separate registration string to append imports and `include_router` once, without replaying an older whole `main.py`. On a rerun, the dictionary is your explicit replacement for those files; keep it synchronized with edits made in the file editor. `apply_student_edits` validates paths/syntax; it does not generate a solution.

### Working with documentation

| Need | Reference |
|---|---|
| Route registration | The visible `include_router` pattern just used |
| Request/response agreement | `app/schemas.py` in your working project |
| Supported question choices | The synthetic bank printed below |
| Framework route organization | [FastAPI bigger applications](https://fastapi.tiangolo.com/tutorial/bigger-applications/) |

<details><summary>Hint 1 — distinguish the failure</summary>Missing endpoint, wrong prompt binding and unhelpful generated wording require different evidence.</details>
<details><summary>Hint 2 — locate the pattern</summary>Follow the new module's router into `include_router`; follow its own `SYSTEM_PROMPT` into `run_diagnosis`.</details>
<details><summary>Hint 3 — diagnose a poor handover</summary>Check what dispatch cannot find or verify. Revise the instruction that addresses that need, then compare two supported questions again.</details>

In [ ]:
handover_edits = {}  # {'app/handover_routes.py': 'complete source'}
apply_student_edits(project, handover_edits)
handover_registration = ''  # Your router import and include_router lines.
if handover_registration.strip(): register_source(project,handover_registration)
case_bank = load_json(REPO, 'examples/patterns/handover_requests.json')
print(json.dumps(case_bank, indent=2))
selected_cases = []  # Choose two distinct names from this bank.
handover_note = ''   # Why these cases, your prompt decision and the evidence you observed.

### Checkpoint — observe wiring before evaluating wording

Run after saving your extension. `passed` covers local wiring and the existing schema. The two selected questions should test different handover needs. The check stays pending if the route file is absent; a pending task does not prevent the later worked examples from running.

Actual HTTP proves the route can be called. Mocked provider-boundary observations prove prompt wiring. Only actual generated responses, reviewed against the documents, can support your judgment about prompt quality.

In [ ]:
handover_report = {'passed':False, 'status':'pending'}
if (project/'app/handover_routes.py').exists():
    handover_report = inspect_route_bindings(project, {'/v1/diagnose':'app.routes', '/v1/dispatch-handover':'app.handover_routes'})
    with demo_service(project) as service:
        handover_http = [{'path':p,'status':httpx.post(service.base_url+p,json=accepted).status_code} for p in ['/v1/diagnose','/v1/dispatch-handover']]
    table(handover_http)
    handover_report['passed'] = handover_report['passed'] and all(row['status']==200 for row in handover_http)
    handover_report['preservation'] = preserved(project,initial)
    handover_report['passed'] = handover_report['passed'] and all(handover_report['preservation'].values())
    handover_report['http'] = handover_http
    handover_report['source_revision'] = revision(project)
print('Handover wiring:', handover_report.get('passed'), handover_report.get('errors',[]))
evidence['handover'] = handover_report

### Generated-answer comparison — required evidence when provider access is available

This branch is off by default. Once local checks pass, set the course-approved model and select two distinct cases above. Enable the branch to make real provider requests; the helper asks for a hidden key and holds it only in the running process configuration. Never enter the key as notebook text.

The four outputs should retain the shared response agreement while serving their different audiences. Compare each factual claim against the supplied documents, revise if needed and record an excerpt in `handover_note`. If access is unavailable, leave this explicitly pending; demo output does not replace it.
This comparison is required to finish the independent practice. Without provider access, keep generated-answer evidence pending and use demo mode only for rehearsal.


In [ ]:
RUN_HANDOVER_LIVE = False
HANDOVER_MODEL = ''
handover_live = {'status':'environment_pending','results':[]}
if RUN_HANDOVER_LIVE:
    assert handover_report['passed'] and fresh(handover_report,project), 'Run current wiring checks first.'
    assert len(selected_cases)==2 and len(set(selected_cases))==2 and all(x in case_bank for x in selected_cases)
    with provider_service(project,models={'OPENROUTER_MODEL':HANDOVER_MODEL}) as service:
        for name in selected_cases:
            for path in ['/v1/diagnose','/v1/dispatch-handover']:
                response = httpx.post(service.base_url+path,json=case_bank[name],timeout=40)
                assert response.status_code==200 and response.json()['mode']=='live' and response.json()['status']=='ready'
                handover_live['results'].append({'case':name,'path':path,'body':response.json()})
    handover_live.update(status='captured_review_required',source_revision=revision(project))
    table(handover_live['results'])
print('Generated handover evidence:', handover_live['status'])
evidence['handover_live'] = handover_live

## 4. Worked example — make the agreement explicit

A **contract** is the agreement callers rely on: allowed inputs and predictable outputs. A small technician form permits at most `300` characters after trimming. Declare that input in the lab while keeping the original route unchanged.

`extra='forbid'` rejects unexpected fields. `equipment_id` remains optional; when supplied it must be a nonempty string of at most `64` characters. The response retains `answer`, `status`, `citations` and `mode`.

**Expected:** the normal request succeeds; an over-limit or malformed request fails before adapter work. A deliberately malformed output is detected after work. A valid-shaped unsupported answer can still pass structural validation—shape is not truth.

In [ ]:
contract_source = """from pydantic import BaseModel, ConfigDict, Field

class TechnicianNoteRequest(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    question: str = Field(min_length=1, max_length=300)
    equipment_id: str | None = Field(default=None, min_length=1, max_length=64)
"""
write_source(lab, 'app/contract_schemas.py', contract_source)
write_source(lab, 'app/contract_routes.py', (REPO/'examples/patterns/contract_routes.py').read_text())
registration = '\nfrom app.contract_routes import router as contract_router\napp.include_router(contract_router)\n'
register_source(lab, registration)
print(contract_source)

In [ ]:
with demo_service(lab) as service:
    for label, payload in [('accepted',accepted),('over_limit',dict(accepted,question='filter '+'x'*294)),('missing_context',incomplete)]:
        response = httpx.post(service.base_url+'/practice/technician-note',json=payload)
        print(label, response.status_code)
examples = load_json(REPO, 'examples/patterns/contract_output_examples.json')
contract_report = inspect_contract(lab,route='/practice/technician-note',route_module='app.contract_routes',max_length=300,accepted_payload=accepted,incomplete_payload=incomplete,output_examples=examples)
assert contract_report['passed'], contract_report['errors']
table(contract_report['cases'])
print('Shape checks only:', contract_report['output_examples'])
evidence['contract_lab'] = contract_report

### Apply the mental model

The intentionally flawed claim is: “Our response passed schema validation, so the equipment has definitely been inspected.” Use the output examples to identify what validation establishes and what evidence is still missing. Write one diagnosis and one fix below. You will apply a different numerical input constraint in your own versioned route next, in the same notebook.

<details><summary>Solution</summary>

**Result you should get:** validation establishes shape and declared constraints, not the truth of an inspection claim. **How:** compare the valid-but-unsupported fixture with the supplied service documents; require claims to be supported and keep unknown conditions explicit in the prompt and answer review.

</details>

In [ ]:
contract_note = ''  # Your diagnosis and the evidence needed beyond valid JSON.

## 5. Worked example — keep two versions independent

A **versioned route** lets an existing caller keep its agreement while a new caller adopts a changed one. A different path is useful only if the new prompt/model binding can change without altering the old behavior.

In the lab, register `/v2/diagnose` with its own prompt and `FIELDCARE_DIAGNOSE_V2_MODEL`. The original keeps `OPENROUTER_MODEL`. Run both routes, then change only the pilot's two-bullet instruction to three bullets. Demo output remains fixed; a mocked boundary probe reports prompt hashes and model selections.

**Check:** both paths remain callable, the old prompt hash stays the same, and the pilot hash changes. This demonstrates configuration independence inside one service, not independent hosting.

In [ ]:
v2_original = (REPO/'examples/patterns/diagnose_v2_routes.py').read_text()
v2_changed = v2_original.replace('as two short bullets.', 'as three short bullets.')
write_source(lab,'app/diagnose_v2_routes.py',v2_original,expected=v2_changed)
registration = '\nfrom app.diagnose_v2_routes import router as diagnose_v2_router\napp.include_router(diagnose_v2_router)\n'
register_source(lab,registration)
print(source_excerpt(lab,'app/diagnose_v2_routes.py'))
lab_versions = {'/v1/diagnose':{'module':'app.routes','model_env':'OPENROUTER_MODEL'}, '/v2/diagnose':{'module':'app.diagnose_v2_routes','model_env':'FIELDCARE_DIAGNOSE_V2_MODEL'}}
version_before = inspect_version_bindings(lab,versions=lab_versions,changed_route='/v2/diagnose',accepted_payload=accepted)
write_source(lab,'app/diagnose_v2_routes.py',v2_changed,expected=v2_original)
version_after = inspect_version_bindings(lab,versions=lab_versions,changed_route='/v2/diagnose',accepted_payload=accepted)
assert version_before['passed'] and version_after['passed']
before_hashes = {row['path']:row['observed_prompt_sha256'] for row in version_before['rounds'][0]['routes']}
after_hashes = {row['path']:row['observed_prompt_sha256'] for row in version_after['rounds'][0]['routes']}
assert before_hashes['/v1/diagnose']==after_hashes['/v1/diagnose']
assert before_hashes['/v2/diagnose']!=after_hashes['/v2/diagnose']
table(version_before['rounds'][0]['routes'])
table(version_after['rounds'][0]['routes'])
with demo_service(lab) as service:
    assert all(httpx.post(service.base_url+p,json=accepted).status_code==200 for p in lab_versions)
evidence['version_lab'] = {'before':version_before,'after':version_after}

### Deprecation is a notice, not removal

**Deprecation** marks an operation as one callers should migrate away from. In this synthetic migration record, a pilot console has moved to `/v2`, but an offline tablet still depends on `/v1`. That does not justify removing `/v1`.

Temporarily mark the old lab route `deprecated=True`. Expect its API documentation flag to change while both routes still return `200`. The `finally` block restores only this demonstrated metadata change.

In [ ]:
original_route = (REPO/'examples/fieldcare/app/routes.py').read_text()
deprecated = original_route.replace('@router.post("/v1/diagnose", response_model=DiagnosticResponse)', '@router.post("/v1/diagnose", response_model=DiagnosticResponse, deprecated=True)')
write_source(lab,'app/routes.py',deprecated,expected=original_route)
try:
    with demo_service(lab) as service:
        api = httpx.get(service.base_url+'/openapi.json').json()
        print('Old route deprecated:',api['paths']['/v1/diagnose']['post']['deprecated'])
        assert all(httpx.post(service.base_url+p,json=accepted).status_code==200 for p in lab_versions)
finally:
    write_source(lab,'app/routes.py',original_route,expected=deprecated)

### Your turn — introduce the dispatch pilot agreement

The pilot form permits `1–600` trimmed question characters. Keep optional bounded `equipment_id`, forbidden extra fields, the existing response shape and clarification behavior. Existing dispatch callers must retain `/v1/dispatch-handover` and its `2000`-character agreement.

Create `HandoverV2Request` in `app/handover_v2_schemas.py`, a `/v2/dispatch-handover` route in `app/handover_v2_routes.py`, and its registration. Give it a separate compact-dispatch prompt and `MODEL_NAME` reading `FIELDCARE_HANDOVER_V2_MODEL`. Reuse the service/adapter.

A synthetic scheduling client still sends valid `900`-character questions to `/v1` and has not verified migration. Explain why removing that route would break its agreement. Choose a supported question and valid missing-context input; determine what proves the new limit and old behavior.

<details><summary>Hint — separate the agreements</summary>The new request class changes the pilot limit; it must not replace the original request class. The new module owns its prompt/model. Both routers must remain registered.</details>

Run the preparation cell before editing. It records the incoming state only after your handover wiring passes. If handover is unfinished, this task stays pending; no answer is substituted for your work.

In [ ]:
version_base = None
if handover_report.get('passed') and fresh(handover_report,project):
    version_base = baseline(project,'versioning',['/v1/diagnose','/v1/dispatch-handover'])
else:
    print('Pending: complete or rerun the current dispatch handover check before applying its versioned contract.')

In [ ]:
version_edits = {}  # Your new schema and route.
version_registration = ''  # Your router import and include_router lines.
if version_base is not None:
    apply_student_edits(project,version_edits)
    if version_registration.strip(): register_source(project,version_registration)
VERSION_READY = False
version_request = None
version_incomplete = None
version_note = ''

In [ ]:
version_report = {'passed':False,'status':'pending'}
if VERSION_READY and version_base is not None:
    version_report = observe_change(project,route='/v2/dispatch-handover',route_module='app.handover_v2_routes',model_env='FIELDCARE_HANDOVER_V2_MODEL',max_length=600,
        accepted_requests=[version_request],incomplete_request=version_incomplete,inherited=version_base,
        bindings={'/v1/dispatch-handover':{'module':'app.handover_routes','model_env':'OPENROUTER_MODEL'}})
    table(version_report['http'])
    print('Original routes:',version_report['inherited_statuses'])
    with demo_service(project) as service:
        scheduling_request = dict(version_request,question='filter '+'x'*893)
        compatibility = {path:httpx.post(service.base_url+path,json=scheduling_request).status_code
                         for path in ['/v1/dispatch-handover','/v2/dispatch-handover']}
    print('Existing 900-character caller:',compatibility)
    version_report['existing_caller'] = compatibility
    version_report['passed'] = version_report['passed'] and compatibility=={'/v1/dispatch-handover':200,'/v2/dispatch-handover':422}
print('Version application:',version_report['passed'])
evidence['version_application'] = version_report

## 6. Supervisor service checkpoint

✂️ **Independent assessment**

HelioDesk's shift supervisor needs a short briefing for **one equipment record**: documented HX filter/airflow checks, unknowns and what the technician must verify next. This is an unseen audience/contract combination, not whole-shift aggregation; no live ticket history or inspection events exist in the supplied evidence.

**Task A — extend:** choose a new POST route, supervisor prompt and independent uppercase model environment name. Use the existing service and adapter; do not hardcode a response.

**Task B — define the agreement:** accept a trimmed `question` of `1–1000` characters, keep optional `equipment_id` bounded to `1–64` characters when present, reject extra fields and preserve `DiagnosticResponse`. Choose two distinct supported requests and one valid missing-context request.

**Task C — expose a version:** register the new operation at a versioned path. A new operation may start at `/v1/<your-name>`; two new versions are not required. Preserve every inherited route, prompt, schema and data file, allowing additions to registration.

### Quick diagnosis

Before editing, use an existing diagnostic request to annotate its route, input, output, adapter location and evidence limitations. Your earlier trace provides the same tools; this diagnosis is not the supervisor solution.

<details><summary>Solution — public diagnostic route only</summary>

**Result you should get:** `POST /v1/diagnose` accepts `question` and optional `equipment_id`, returns `answer`, `status`, `citations`, `mode`, and calls through `run_diagnosis` to `generate_answer` when the supported context is present. **How:** inspect the route/schema/service source and compare the actual HTTP result with the separate function-entry probe. Demo success proves neither generated wording nor equipment condition.

</details>

Record the incoming state, then implement independently. Course patterns and official documentation are available references; follow your instructor's stated rules for collaboration/submission. No grading weights or closed-book policy are implied here.

In [ ]:
assessment_base = None
if version_report.get('passed') and fresh(version_report,project):
    with demo_service(project) as service:
        api = httpx.get(service.base_url+'/openapi.json').json()
        inherited_paths = sorted(p for p,v in api['paths'].items() if 'post' in v)
    assessment_base = baseline(project,'assessment',inherited_paths)
    print('Preserve:',assessment_base['inherited_paths'])
else:
    print('Pending: complete or rerun the current versioned dispatch check first.')
diagnosis_note = ''

In [ ]:
assessment_edits = {}  # Your new implementation files; empty preserves existing files.
assessment_registration = ''  # Your router import and include_router lines.
if assessment_base is not None:
    apply_student_edits(project,assessment_edits)
    if assessment_registration.strip(): register_source(project,assessment_registration)
ASSESSMENT_READY = False
new_route = ''
route_module = ''
model_env = ''
assessment_requests = []  # Two distinct supported request mappings.
assessment_incomplete = None
assessment_note = ''      # Contract, version choice, prompt reasoning and evidence limitations.

In [ ]:
assessment_report = {'passed':False,'status':'pending'}
if ASSESSMENT_READY and assessment_base is not None:
    assert len(assessment_requests)==2 and assessment_requests[0]!=assessment_requests[1]
    assessment_report = observe_change(project,route=new_route,route_module=route_module,model_env=model_env,max_length=1000,
        accepted_requests=assessment_requests,incomplete_request=assessment_incomplete,inherited=assessment_base,
        bindings={'/v1/diagnose':{'module':'app.routes','model_env':'OPENROUTER_MODEL'},
                  '/v1/dispatch-handover':{'module':'app.handover_routes','model_env':'OPENROUTER_MODEL'},
                  '/v2/dispatch-handover':{'module':'app.handover_v2_routes','model_env':'FIELDCARE_HANDOVER_V2_MODEL'}})
    table(assessment_report['http'])
    print('Retained routes:',assessment_report['inherited_statuses'])
print('Local structural checks:',assessment_report['passed'],'— instructor review still required.')
evidence['assessment'] = assessment_report

### Collect real briefing evidence when access is ready

The local checks verify contracts and wiring. To assess the actual supervisor briefing, run the two selected supported requests with the course-approved provider model and review the generated wording against the documents. Quote one observed strength/limitation in `generated_review`.

Leave the branch off if the environment is unavailable. The checkpoint will retain that limitation explicitly. A successful live request is still not an instructor grade.

In [ ]:
RUN_ASSESSMENT_LIVE = False
ASSESSMENT_MODEL = ''
generated_review = ''
assessment_live = {'status':'environment_pending','results':[]}
if RUN_ASSESSMENT_LIVE:
    selection = {'route':new_route,'route_module':route_module,'model_env':model_env,'max_length':1000,'accepted_requests':assessment_requests,'incomplete_request':assessment_incomplete}
    assert assessment_report['passed'] and fresh(assessment_report,project,selection), 'Rerun current structural checks.'
    with provider_service(project,models={'OPENROUTER_MODEL':ASSESSMENT_MODEL,model_env:ASSESSMENT_MODEL}) as service:
        for payload in assessment_requests:
            response = httpx.post(service.base_url+new_route,json=payload,timeout=40)
            assert response.status_code==200 and response.json()['mode']=='live' and response.json()['status']=='ready'
            assessment_live['results'].append({'request':payload,'body':response.json()})
    assessment_live.update(status='captured_review_required',source_revision=revision(project),selection=json.loads(json.dumps(selection)))
    table(assessment_live['results'])
print('Generated briefing evidence:',assessment_live['status'])
evidence['assessment_live'] = assessment_live

## 7. Save one sprint checkpoint

Inspect the current source and evidence before sharing. The exporter excludes environment files, but cannot protect a key you pasted into an allowed Python/JSON file. Save your personal notebook separately to retain notes/cell edits; download the source ZIP before Colab's temporary files disappear.

**Final check:** the service has the original diagnostic route, your handover and its pilot version, and your new supervisor operation. Compare the declared input limits, inspect accepted/rejected evidence, and verify prior contracts still work. Record pending generated-answer review honestly.

If source or request selections change after observations, rerun the relevant checks. The final record keeps structural readiness and generated-answer review separate. The notebook exports useful incomplete work too; it never labels an untouched exercise complete.

In [ ]:
from module_b.live_sessions import read_records
assessment_report = globals().get('assessment_report', {'passed':False, 'status':'pending'})
assessment_live = globals().get('assessment_live', {'status':'environment_pending', 'results':[]})
current_selection = {'route':globals().get('new_route',''), 'route_module':globals().get('route_module',''),
    'model_env':globals().get('model_env',''), 'max_length':1000,
    'accepted_requests':globals().get('assessment_requests',[]),
    'incomplete_request':globals().get('assessment_incomplete')}
structural_ready = bool(assessment_report.get('passed') and fresh(assessment_report,project,current_selection))
readiness = {'structural_ready':structural_ready,
    'generated_answer_status':assessment_live['status'] if not assessment_live['results'] or fresh(assessment_live,project,current_selection) else 'stale_rerun_required',
    'written_evidence_present':all(str(globals().get(name,'')).strip() for name in ['diagnosis_note','assessment_note','generated_review'])}
evidence['live_sessions'] = read_records(project)
SPRINT1_EXPORT = export_progress(project,REPO/'artifacts',sprint=1,evidence=evidence,
    notes={name:globals().get(variable,'') for name,variable in [('handover','handover_note'),('contracts','contract_note'),('versioning','version_note'),('diagnosis','diagnosis_note'),('assessment','assessment_note'),('generated_review','generated_review')]},readiness=readiness)
print('Readiness:',readiness)


## Watch out for

| Symptom | Check and recovery |
|---|---|
| New file but route returns `404` | Check the router import and `include_router`, then run a new local process. |
| Demo wording ignores a prompt edit | Expected; inspect binding evidence, then collect real responses when available. |
| Missing context is rejected as malformed | Check that `equipment_id` remains optional and distinguish `422` from clarification. |
| Changing `/v2` changes `/v1` | Look for shared request classes or prompt/model bindings where separate ones were required. |
| A worked write refuses changed source | Preserve your edit; use a new deliberate `RUN_NAME` for a clean worked lab. |
| An exercise says pending | Complete its visible source/selection cells; do not treat Run all as completion. |
| Files disappear after runtime reset | Upload the saved source ZIP and set `RESUME_ZIP`; your notebook alone does not contain edited service files. |

## Quick reference

| Operation | Purpose |
|---|---|
| `source_excerpt` | Inspect actual working Python definitions |
| `httpx.get` / `httpx.post` | Call the owned service with an explicit method/path/body |
| `write_source` / `register_source` | Guard worked source writes and append visible registration once |
| `apply_student_edits` | Save your explicitly chosen file contents |
| `inspect_contract` / `inspect_version_bindings` | Observe separate no-provider validation and binding behavior |
| `observe_change` | Check a new operation and preserve inherited agreements |
| `export_progress` | Save current source, evidence, notes and honest readiness |

## Summary

You turned FieldCare into a service other systems can call and adapt to: explicit contracts, purpose-specific prompts and versioned agreements that preserve existing callers. Your saved checkpoint is the service to protect in the next sprint.